# Brain Tumor MRI Classification — ViT Fine-Tuning
**Prepared by Kawchar Husain**

Model: `google/vit-base-patch16-224` fine-tuned on Brain Tumor MRI Dataset  
Dataset: kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset  
Hardware: Kaggle T4 GPU (approx. 40 min for 6 epochs)

> **Before running:** Enable Internet and T4 GPU in Settings → Session Options.

## 0. Setup & Imports

In [ ]:
import os, json, torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from transformers import (
    ViTForImageClassification,
    ViTImageProcessor,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import confusion_matrix, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 1. Dataset Paths

In [ ]:
TRAIN_DIR = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training'
TEST_DIR  = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing'

# Verify directories exist
assert Path(TRAIN_DIR).exists(), f'Training dir not found: {TRAIN_DIR}'
assert Path(TEST_DIR).exists(),  f'Testing dir not found:  {TEST_DIR}'
print('Directories verified.')

## 2. Class Distribution (EDA)

In [ ]:
# Count images per class in Training folder
class_counts = {}
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.jpg','.jpeg','.png'))])
        class_counts[cls] = count

print('Training class distribution:')
for cls, cnt in class_counts.items():
    print(f'  {cls:12s}: {cnt}')
print(f'  Total        : {sum(class_counts.values())}')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(class_counts.keys(), class_counts.values(),
              color=['#E74C3C','#3498DB','#2ECC71','#9B59B6'],
              edgecolor='white', linewidth=0.8, alpha=0.9)
ax.set_title('Class Distribution — Training Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Tumor Class'); ax.set_ylabel('Number of Images')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Custom Dataset Class

In [ ]:
CHECKPOINT  = 'google/vit-base-patch16-224'
NUM_CLASSES = 4

image_processor = ViTImageProcessor.from_pretrained(CHECKPOINT)

class BrainTumorDataset(Dataset):
    """Wraps torchvision.ImageFolder and applies ViTImageProcessor per item."""
    def __init__(self, root, processor):
        self.data      = datasets.ImageFolder(root)
        self.processor = processor
        self.classes   = self.data.classes

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image, label = self.data[idx]
        inputs = self.processor(
            images=image.convert('RGB'),
            return_tensors='pt'
        )
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'labels':       torch.tensor(label),
        }

train_dataset = BrainTumorDataset(TRAIN_DIR, image_processor)
test_dataset  = BrainTumorDataset(TEST_DIR,  image_processor)
CLASS_NAMES   = train_dataset.classes

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

print('Classes:      ', CLASS_NAMES)
print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')

## 4. Model — Load Pretrained ViT

In [ ]:
# ignore_mismatched_sizes=True replaces the 1000-class ImageNet head with 4 classes
model = ViTForImageClassification.from_pretrained(
    CHECKPOINT,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model: {CHECKPOINT}')
print(f'Total parameters: {total_params:,}')

## 5. Training

In [ ]:
EPOCHS = 6
LR     = 2e-5

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = total_steps // 10

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

history = {'loss': [], 'acc': []}

for epoch in range(EPOCHS):
    # ── Training phase ──
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch   = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss    = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    # ── Evaluation phase ──
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            batch   = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            preds   = outputs.logits.argmax(dim=-1)
            correct += (preds == batch['labels']).sum().item()
            total   += batch['labels'].size(0)

    avg_loss = total_loss / len(train_loader)
    acc      = correct / total
    history['loss'].append(avg_loss)
    history['acc'].append(acc)
    print(f'Epoch {epoch+1}/{EPOCHS}  Loss: {avg_loss:.4f}  Acc: {acc:.4f}')

## 6. Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(epochs_range, history['loss'], marker='o', color='#E74C3C', linewidth=2)
axes[0].set_title('Training Loss per Epoch', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Average Loss')
axes[0].set_xticks(epochs_range)

axes[1].plot(epochs_range, history['acc'], marker='o', color='#2ECC71', linewidth=2)
axes[1].set_title('Test Accuracy per Epoch', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05); axes[1].set_xticks(epochs_range)
for x, y in zip(epochs_range, history['acc']):
    axes[1].text(x, y + 0.01, f'{y:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\nFinal Test Accuracy: {history["acc"][-1]:.4f}')

## 7. Confusion Matrix

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        preds  = model(**batch).logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Confusion Matrix — Test Set', fontsize=13)
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))
print(f'\nFinal Test Accuracy: {sum(p==l for p,l in zip(all_preds,all_labels))/len(all_labels):.4f}')

## 8. Save Artifacts

In [ ]:
os.makedirs('artifacts', exist_ok=True)

# Save model weights only (not the full object) for portability across Python/transformers versions
torch.save(model.state_dict(), 'artifacts/vit_brain_tumor.pt')
print('Model weights saved: artifacts/vit_brain_tumor.pt')

# Save class names in the same order as ImageFolder
# ImageFolder sorts alphabetically: ['glioma', 'meningioma', 'notumor', 'pituitary']
with open('artifacts/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)
print('Class names saved: artifacts/class_names.json')
print('Classes:', CLASS_NAMES)
print('\nDownload both files from the Kaggle Output panel')
print('and place them in artifacts/ on your local machine.')